# 멋사 중부대 14기 · LoRA 실습 노트북

같은 질문을 **기본 모델**과 **LoRA로 학습한 모델**에 넣고, 입력 · 출력 · 점수가 어떻게 달라지는지 단계별로 확인합니다.
셀은 위에서부터 순서대로 실행하세요. 셀 왼쪽의 ▶ 버튼 또는 `Ctrl+Enter`로 실행합니다.
수업에서는 웹 실습 가이드의 코드 블록을 Colab 새 노트북에 하나씩 붙여 넣어 진행합니다. 이 파일은 같은 코드를 한 번에 모아 둔 것입니다.

| 단계 | 할 일 | 화면에서 확인할 것 |
| --- | --- | --- |
| 1 | Colab 환경 세팅 | GPU, 패키지 버전, 데이터 구성표 |
| 2 | 기본 모델에 질문하기 | 입력 → 출력 JSON → 채점 카드, 검증 점수 막대 |
| 3 | 기본 변수로 LoRA 학습 | 학습 파라미터 비율, loss 그래프 |
| 4 | LoRA 모델에 같은 질문하기 | 입력 → 출력 JSON → 채점 카드, 검증 점수 막대 |
| 5 | 최종 출력 비교 | 질문별 학습 전후 비교표, 지표 막대 |
| 6 | **과제**: 나만의 출력 학습 | 내 문장 정확 일치, 실험 순위표 |

> **시작 전에** 실습 사이트에서 `lora-workshop.zip`을 내려받고, 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하고 저장하세요.
> 실습 데이터의 정규 세미나 시간과 운영진 정보는 실제 14기 정보이고, 나머지 규정·일정·금액은 교육용 예시입니다.

---
## 1단계 · Colab 환경 세팅
GPU를 확인하고, 실습 파일을 올려 패키지를 설치합니다. 설치가 끝나면 **세션을 한 번 다시 시작**해야 합니다.

In [ ]:
#@title 1-1. GPU 확인
# 이 실습은 NVIDIA GPU가 필요합니다. 'Tesla T4' 같은 이름이 보이면 정상입니다.
# 오류가 나면: 메뉴 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 이 셀을 다시 실행하세요. (TPU는 이 실습에서 쓸 수 없습니다)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 1-2. 실습 파일 올리기
# 실습 사이트에서 내려받은 lora-workshop.zip을 Colab에 올리고 압축을 풉니다.
# 실행하면 아래에 '파일 선택' 버튼이 나타납니다 → lora-workshop.zip 선택.
import os
os.chdir("/content")
!rm -rf lora-workshop.zip            # 전에 올린 ZIP이 있으면 지움(이름이 'lora-workshop (1).zip'이 되는 것 방지)
from google.colab import files
files.upload()
!unzip -o -q lora-workshop.zip       # lora-workshop/ 폴더에 압축 해제
%cd /content/lora-workshop
!ls

In [ ]:
#@title 1-3. 패키지 설치 (3~5분)
# unsloth      : LoRA 학습을 빠르고 메모리 적게 하도록 돕는 라이브러리
# transformers : 모델·토크나이저 불러오기      trl : 지도 미세조정(SFT) 학습 도구
# peft         : LoRA 어댑터(A·B 행렬) 구현    datasets : 학습 데이터 형식
%cd /content/lora-workshop
!pip install -q -r requirements.txt
print("설치 완료 → 메뉴에서 '런타임 → 세션 다시 시작'을 누른 뒤 1-4부터 실행하세요.")

In [ ]:
#@title 1-4. (세션 다시 시작 후) 실습 코드 불러오기와 데이터 확인
# 세션을 다시 시작하면 그 전에 만든 변수가 모두 지워집니다. 그래서 이 셀부터 다시 실행합니다.
import os
if not os.path.exists("/content/lora-workshop/train.py"):
    raise RuntimeError("실습 파일이 없습니다. 먼저 1-2 셀에서 lora-workshop.zip을 올리세요.")
%cd /content/lora-workshop
import importlib.metadata as md
# 버전만 확인합니다. 여기서 transformers를 import하면 Unsloth 최적화가 빠지므로 import하지 않습니다.
try:
    for name in ["torch", "transformers", "trl", "peft", "unsloth"]:
        print(f"{name:12s} {md.version(name)}")
except md.PackageNotFoundError as exc:
    raise RuntimeError(f"{exc.name} 패키지가 없습니다. 1-3 설치를 실행하고 세션을 다시 시작하세요.") from None

import train as lab        # 모델 불러오기 · 질문 · 학습 · 평가 함수 (train.py)
import lab_view as view    # 표 · 카드 · 그래프로 보여 주는 함수 (lab_view.py)
docs, splits = lab.load_data()     # 데이터를 읽으면서 형식·중복·분할 누수를 함께 검사합니다
view.show_data_overview(docs, splits)

---
## 2단계 · 기본 모델에 질문하고 평가하기
**2-1 하나를 실행하면** 원본 Qwen3.5-2B를 불러오고, 검증(validation) 10문항에 답한 뒤 점수를 보여 줍니다. 웹 가이드 입력 1~5의 답변 카드와 표도 같은 결과에서 골라 보여 줍니다.
모델에 들어가는 입력은 `시스템 지시문 + 참고 문서 + 질문`이고, 출력은 JSON 하나여야 합니다. 직접 질문하고 싶으면 선택 셀 **2-2**를 실행하세요.


In [ ]:
#@title 2-1. 기본 모델 질문·평가 (모델 불러오기부터 결과 확인까지)
# 1-4에서 실습 코드를 불러온 뒤 실행하세요.
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("view", "1-4(실습 코드 불러오기)")]:
    if globals().get(_need) is None:
        raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")

# 원본 모델을 불러옵니다. 처음에는 다운로드 때문에 2~4분이 걸릴 수 있습니다.
model, tokenizer, model_info = lab.load_base_model()
view.show_model_info(model_info)

# 검증 10문항에 한 번씩만 답하고, 그 결과를 카드·표·점수에 함께 씁니다.
base_val = lab.evaluate(model, tokenizer, split="validation", mode="base")
for result in base_val:
    result["dataset_split"] = "validation"
_base_by_id = {result["id"]: result for result in base_val}
base_guide = [_base_by_id[item["row_id"]] for item in lab.load_guide()]
for result in base_guide:
    view.show_answer_card(result)
view.show_results_table(base_guide, "기본 모델 · 가이드 입력 5개")
view.show_metrics(lab.summarize(base_val), "기본 모델 · 검증 10문항 점수")
view.show_results_table(base_val, "기본 모델 · 검증 10문항")


In [ ]:
#@title 2-2. (선택) 직접 질문해 보기
질문 = "14기 대표는 누구야?"  #@param {type:"string"}
문서_ID = "T11"  #@param ["T01", "T02", "T03", "T04", "T05", "T06", "T07", "T08", "T09", "T10", "T11", "T12", "V01", "V02", "V03", "V04", "F01", "F02", "F03", "F04", "직접 입력"]
직접_입력_문서 = ""  #@param {type:"string"}
# 오른쪽 입력 칸에 질문을 쓰고 문서_ID를 고른 뒤 실행합니다. 정답은 입력하지 않습니다.
# 직접_입력_문서는 문서_ID를 '직접 입력'으로 골랐을 때만 씁니다(평소에는 비워 둠).
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("view", "1-4(실습 코드 불러오기)"), ("tokenizer", "2-1(기본 모델 질문·평가)"), ("model", "2-1(기본 모델 질문·평가)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
답변 = lab.ask(model, tokenizer, 질문, 문서_ID, 직접_입력_문서, mode="base")
view.show_answer_card(답변)

---
## 3단계 · 제공 데이터와 기본 변수로 LoRA 학습
기본 모델의 가중치는 **고정**하고, 층마다 작은 행렬 A·B(LoRA 어댑터)만 학습합니다.
이 단계에서는 변수를 바꾸지 않습니다. 변수 바꾸기는 6단계 과제에서 합니다.

In [ ]:
#@title 3-1. 학습 설정 확인 (기본값 그대로)
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("splits", "1-4(실습 코드 불러오기)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
LEARNING_RATE = lab.LEARNING_RATE   # 2e-4 : 한 번 업데이트할 때 움직이는 크기
EPOCHS = lab.EPOCHS                 # 2    : 학습 데이터 36개를 몇 바퀴 볼지
LORA_RANK = lab.LORA_RANK           # 8    : 추가 행렬 A·B의 폭 (alpha = 2 × rank)
view.show_settings(LEARNING_RATE, EPOCHS, LORA_RANK, len(splits["train"]))

In [ ]:
#@title 3-2. 기본 모델에 LoRA 어댑터 붙이기
# 붙인 직후에는 B 행렬이 0이라 출력이 기본 모델과 같습니다. 학습하면서 A·B가 바뀝니다.
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("model", "2-1(기본 모델 질문·평가)"), ("LORA_RANK", "3-1(학습 설정 확인)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
model, lora_info = lab.add_lora(model, LORA_RANK)
view.show_lora_info(lora_info)

In [ ]:
#@title 3-3. LoRA 학습 실행 (T4 기준 처음 4~6분)
# 한 스텝 = 예시 4개로 순전파 → loss 계산 → 역전파 → AdamW 업데이트.
# 학습 loss는 매 스텝, 검증 loss는 epoch마다 기록해 그래프로 그립니다.
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("model", "2-1(기본 모델 질문·평가)"), ("lora_info", "3-2(LoRA 어댑터 붙이기)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
train_info = lab.train_lora(model, tokenizer, splits, LEARNING_RATE, EPOCHS)
view.show_training(train_info)

---
## 4단계 · LoRA 모델에 같은 질문을 하고 평가하기
**4-1 하나를 실행하면** 3단계에서 학습한 모델이 2단계와 **똑같은 검증 10문항**에 답하고 점수를 보여 줍니다. 가이드 입력 1~5의 카드와 표도 이 결과를 다시 사용합니다.
지시문 · 문서 · 생성 설정은 같고, 달라진 것은 LoRA 어댑터뿐입니다. **4-2**에서는 2-2에 넣었던 질문을 직접 다시 물어볼 수 있습니다.


In [ ]:
#@title 4-1. LoRA 모델 질문·평가 (학습한 모델로 결과 확인)
# 3-3에서 학습을 마친 모델을 그대로 씁니다.
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("view", "1-4(실습 코드 불러오기)"), ("model", "3-3(LoRA 학습 실행)"), ("tokenizer", "2-1(기본 모델 질문·평가)"), ("train_info", "3-3(LoRA 학습 실행)")]:
    if globals().get(_need) is None:
        raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")

# 같은 검증 10문항에 한 번씩만 답하고, 가이드 5개도 이 결과에서 고릅니다.
lora_val = lab.evaluate(model, tokenizer, split="validation", mode="lora")
for result in lora_val:
    result["dataset_split"] = "validation"
_lora_by_id = {result["id"]: result for result in lora_val}
lora_guide = [_lora_by_id[item["row_id"]] for item in lab.load_guide()]
for result in lora_guide:
    view.show_answer_card(result)
view.show_results_table(lora_guide, "LoRA 모델 · 가이드 입력 5개")
view.show_metrics(lab.summarize(lora_val), "LoRA 모델 · 검증 10문항 점수")
view.show_results_table(lora_val, "LoRA 모델 · 검증 10문항")


In [ ]:
#@title 4-2. (선택) 직접 질문해 보기
질문 = "14기 대표는 누구야?"  #@param {type:"string"}
문서_ID = "T11"  #@param ["T01", "T02", "T03", "T04", "T05", "T06", "T07", "T08", "T09", "T10", "T11", "T12", "V01", "V02", "V03", "V04", "F01", "F02", "F03", "F04", "직접 입력"]
직접_입력_문서 = ""  #@param {type:"string"}
# 2-2에서 넣었던 질문을 그대로 넣어 비교해 보세요. T로 시작하는 문서는 학습에 쓴 문서입니다.
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("view", "1-4(실습 코드 불러오기)"), ("tokenizer", "2-1(기본 모델 질문·평가)"), ("model", "2-1(기본 모델 질문·평가)"), ("train_info", "3-3(LoRA 학습 실행)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
답변 = lab.ask(model, tokenizer, 질문, 문서_ID, 직접_입력_문서, mode="lora")
view.show_answer_card(답변)

---
## 5단계 · 최종 출력 비교
같은 입력에 대한 학습 전후 출력을 나란히 봅니다. **▲ 개선 / ▼ 악화 / = 그대로**는 종합 성공(형식·정답·근거·status가 모두 맞음) 기준입니다.
점수만 보지 말고 실제 출력 문장이 어떻게 달라졌는지 함께 읽어 보세요. 마지막으로 한 번도 보지 않은 **test 10문항**으로 확인합니다.

In [ ]:
#@title 5-1. 가이드 입력 5개 · 학습 전후 비교
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("base_guide", "2-1(기본 모델 질문·평가)"), ("lora_guide", "4-1(LoRA 모델 질문·평가)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
view.show_comparison(base_guide, lora_guide, "가이드 입력 5개")

In [ ]:
#@title 5-2. 검증 10문항 · 점수와 출력 비교
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("base_val", "2-1(기본 모델 질문·평가)"), ("lora_val", "4-1(LoRA 모델 질문·평가)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
view.show_metrics(lab.summarize(base_val + lora_val), "검증 10문항 · 기본 모델 vs LoRA 모델")
view.show_comparison(base_val, lora_val, "검증 10문항")

In [ ]:
#@title 5-3. test 10문항으로 마지막 확인 (한 번만)
# 학습에도, 비교에도 쓰지 않은 test 문항(F01~F04 문서)에 LoRA 모델이 어떻게 답하는지 마지막으로 봅니다.
# 기본 모델 점수는 2-1·5-2의 검증 결과와 비교해 보세요. (기본 모델은 이미 LoRA가 붙어 다시 잴 수 없음)
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("model", "2-1(기본 모델 질문·평가)"), ("train_info", "3-3(LoRA 학습 실행)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
lora_test = lab.evaluate(model, tokenizer, split="test", mode="lora")
view.show_metrics(lab.summarize(lora_test), "LoRA 모델 · test 10문항 (처음 보는 문서)")
view.show_results_table(lora_test, "LoRA 모델 · test 10문항")

In [ ]:
#@title 5-4. 결과 저장
결과_내려받기 = False  #@param {type:"boolean"}
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("base_val", "2-1(기본 모델 질문·평가)"), ("lora_val", "4-1(LoRA 모델 질문·평가)"), ("lora_info", "3-2(LoRA 어댑터 붙이기)"), ("train_info", "3-3(LoRA 학습 실행)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
saved = lab.save_session(base_val, lora_val, train_info, lora_info, LEARNING_RATE, EPOCHS, LORA_RANK,
                         globals().get("base_guide"), globals().get("lora_guide"))
print("저장 위치:", saved)
if 결과_내려받기:
    !cd /content/lora-workshop/outputs && zip -qr /content/my-lora-result.zip "{saved.name}"
    from google.colab import files
    files.download("/content/my-lora-result.zip")

---
## 6단계 · 과제: 나만의 출력 학습
1~5단계에서는 **답하는 형식**(근거 ID, 답 보류)을 학습했습니다. 과제에서는 **문장 내용 자체**를 학습합니다.
내가 정한 문장 3개를, 문서 없이 질문만 받아도 **글자 그대로** 말하게 만드는 것이 목표입니다.

**목표:** 처음 보는 질문(질문 c)에도 정확히 답하면서, 관계없는 질문에는 내 문장을 엉뚱하게 말하지 않는 **가장 적은 학습 설정** 찾기

1. 6-1 예시 코드의 `"____"` 빈칸을 내 문장과 질문으로 채웁니다(질문 a·b는 학습, 질문 c는 시험용이라 표현을 다르게).
2. 6-2를 **기본값(2e-4 · 2 · 8)으로 먼저** 실행해 결과를 봅니다. 아마 아직 못 외웠을 거예요.
3. 변수를 **한 번에 하나씩** 바꿔 6-2를 다시 실행하고, 6-3 순위표로 비교합니다.

| 변수 | 해 볼 값 | 관찰할 점 |
| --- | --- | --- |
| EPOCHS | 2 · 5 · 10 · 20 | 몇 바퀴부터 글자 그대로 나오는지 |
| LEARNING_RATE | 2e-4 · 5e-4 · 1e-3 | 크게 하면 빨리 외우지만, 관계없는 질문까지 망가지는지 |
| LORA_RANK | 4 · 8 · 16 · 32 | 학습 파라미터를 늘리면 더 빨리 외우는지 |

실험 한 번에 모델 불러오기를 포함해 몇 분이 걸립니다(EPOCHS가 클수록 길어짐). 조원끼리 변수를 나눠 맡으면 빨라요.

In [ ]:
#@title 6-1. 나만의 출력 문장 3개 쓰기 (빈칸 채우기)
# "____" 부분을 지우고 직접 씁니다. 따옴표(" ")는 남겨 두세요.
#   출력   : 모델이 글자 그대로 말하길 원하는 문장 (80자 이하)
#   질문_a·b: 그 출력을 끌어낼 질문 → 학습에 씀
#   질문_c : 같은 뜻을 다른 표현으로 묻는 질문 → 학습에 쓰지 않고 '처음 보는 질문' 시험에만 씀
나의_문장 = [
    {"출력": "멋사 중부대 14기 대표는 백서진입니다.",      # ← 예시 문장. 지우고 내 문장으로 바꿔도 됩니다
     "질문_a": "14기 대표가 누구야?",
     "질문_b": "멋사 중부대 대표 알려줘",
     "질문_c": "이번 기수 대표는 누구니?"},
    {"출력": "____",
     "질문_a": "____",
     "질문_b": "____",
     "질문_c": "____"},
    {"출력": "____",
     "질문_a": "____",
     "질문_b": "____",
     "질문_c": "____"},
]
# 학습과 관계없는 질문: 학습 뒤에도 내 문장을 엉뚱하게 말하지 않는지 확인합니다.
관계없는_질문 = ["파이썬은 어떤 프로그래밍 언어야?", "1 더하기 1은 얼마야?"]

# 빈칸·중복·길이를 검사하고 학습용(질문 a·b)과 시험용(질문 c)으로 나눕니다.
if globals().get("lab") is None: raise RuntimeError("먼저 1-4(실습 코드 불러오기) 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
과제 = lab.make_custom_task(나의_문장, 관계없는_질문)
view.show_custom_task(과제)

In [ ]:
#@title 6-2. 변수 하나 바꿔서 학습하고 확인하기
LEARNING_RATE = 2e-4  #@param {type:"number"}
EPOCHS = 2  #@param {type:"number"}
LORA_RANK = 8  #@param [4, 8, 16, 32] {type:"raw"}
메모 = "기본값"  #@param {type:"string"}
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("과제", "6-1(나만의 출력 문장 쓰기)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
# 앞에서 쓰던 모델을 지워 GPU 메모리를 비웁니다(2~5단계 모델, 이전 실험 모델 모두).
model = tokenizer = None
lab.free_gpu()
# 원본 모델을 새로 불러와 내 문장으로 LoRA를 처음부터 학습합니다(이전 실험은 누적되지 않음).
# 같은 문장으로 처음 실험할 때는 학습 전 기본 모델의 답도 함께 기록합니다.
실험 = lab.run_custom_experiment(과제, LEARNING_RATE, EPOCHS, int(LORA_RANK), label=메모, keep_model=True)
model, tokenizer = 실험.pop("model"), 실험.pop("tokenizer")   # 6-4에서 자유롭게 물어볼 수 있도록 남겨 둠
view.show_custom_result(실험)

In [ ]:
#@title 6-3. 내 문장으로 한 실험 순위 보기
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("과제", "6-1(나만의 출력 문장 쓰기)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
기록 = lab.load_custom_experiments(과제)
view.show_custom_experiments(기록, lab.rank_custom_experiments(기록))

In [ ]:
#@title 6-4. 마지막으로 학습한 모델에 자유롭게 물어보기
질문 = "우리 동아리 대표가 누구였지?"  #@param {type:"string"}
# 문서 없이 질문만 넣습니다. 학습한 문장을 다른 표현으로 물어보거나, 전혀 다른 질문도 해 보세요.
# 앞 단계를 건너뛰었는지 확인합니다(세션을 다시 시작했다면 1-4부터 다시 실행).
for _need, _cell in [("lab", "1-4(실습 코드 불러오기)"), ("model", "6-2(학습하고 확인하기)"), ("실험", "6-2(학습하고 확인하기)")]:
    if globals().get(_need) is None: raise RuntimeError(f"먼저 {_cell} 셀을 실행하세요. 세션을 다시 시작했다면 1-4부터 다시 실행해야 합니다.")
view.show_plain_answer(lab.ask_plain(model, tokenizer, 질문))

### 발표할 내용
| 질문 | 기록할 내용 |
| --- | --- |
| 어떤 문장을 학습시켰나요? | 내 문장 3개와 질문 a·b·c |
| 몇 번째 실험에서 정확히 나왔나요? | 6-3 순위표: 바꾼 변수와 그때의 정확 일치 |
| 너무 많이 학습하면 무엇이 망가졌나요? | 관계없는 질문의 답, 처음 보는 질문(c)의 답 |
| 1~5단계(형식 학습)와 무엇이 다른가요? | 문서를 주고 답하는 방식 vs 문서 없이 문장을 외움 → RAG와 LoRA는 언제 쓸까? |